## load_silver_realtor
Conforms `bronze.realtor_metro_monthly` (all-STRING) into the typed Silver fact `silver.fact_realtor_metro_monthly`. Per-source fact, joined to `dim_geo` directly on `cbsa_code`.

**Transforms:** cast STRING -> typed (USD/size amounts rounded to BIGINT via `round(double,0).cast(long)`; rates/shares/ratios kept DOUBLE); `month_date_yyyymm` -> month-end `date_key`; resolve `geo_key` from `dim_geo`. **No silent drops** — rows with an unmatched CBSA or a malformed numeric value are routed to `silver.quarantine` with a reason (`unmatched_geography` / `cast_failed:<cols>`), carrying the full Bronze row as `raw_payload`.

**Write:** MERGE on `(geo_key, date_key)`. Audit via `StepLog` (pipeline_step_log) + `transform_detail_log` (rows read / inserted / updated / rejected).

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects BRONZE, SILVER, AUDIT, PIPELINE_RUN_ID, STATUS_*, StepLog, Utils,
# transform_detail_log_insert, F, datetime, timezone, spark, dbutils.

STEP_SEQUENCE = 1
SOURCE_SYSTEM = "realtor"
SOURCE_TABLE  = f"{BRONZE}.realtor_metro_monthly"
TARGET_TABLE  = f"{SILVER}.fact_realtor_metro_monthly"
QUARANTINE    = f"{SILVER}.quarantine"
DIM_GEO       = f"{SILVER}.dim_geo"

# Cast spec. BIGINT cols are rounded whole values (USD amounts, counts, sqft); the Bronze
# strings are float-formatted ('529900.0', '878.0'), so cast via double first then round.
BIGINT_COLS = [
    "median_listing_price", "active_listing_count", "new_listing_count",
    "price_increased_count", "price_reduced_count", "pending_listing_count",
    "median_listing_price_per_square_foot", "median_square_feet",
    "average_listing_price", "total_listing_count",
]
DOUBLE_COLS = [
    "median_days_on_market", "price_increased_share", "price_reduced_share", "pending_ratio",
]
NUMERIC_COLS = BIGINT_COLS + DOUBLE_COLS                 # cast-error detection covers these
VALUE_COLS   = BIGINT_COLS + DOUBLE_COLS + ["quality_flag"]   # all non-key fact columns

In [ ]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_realtor: step_log_id={step.step_log_id}")

In [ ]:
# Read Bronze, build the typed staging frame: cast value columns, derive date_key, detect
# per-row cast errors, capture raw_payload, and resolve geo_key from dim_geo.
try:
    bronze = spark.table(SOURCE_TABLE)
    rows_read = bronze.count()

    # month_date_yyyymm ('202604') -> month-end date -> yyyymmdd INT (matches dim_date grain).
    month_end = F.last_day(F.to_date(F.col("month_date_yyyymm"), "yyyyMM"))
    date_key  = (F.year(month_end) * 10000 + F.month(month_end) * 100
                 + F.dayofmonth(month_end)).cast("int")

    bigint_round = lambda col_name: F.round(F.col(col_name).cast("double"), 0).cast("long")
    # A cast error = source non-empty but its double cast is null (genuinely malformed).
    cast_err = lambda col_name: (F.col(col_name).isNotNull()) & (F.trim(F.col(col_name)) != F.lit("")) \
                         & (F.col(col_name).cast("double").isNull())

    typed = bronze.select(
        F.col("cbsa_code"),
        F.col("month_date_yyyymm"),
        date_key.alias("date_key"),
        *[bigint_round(col_name).alias(col_name) for col_name in BIGINT_COLS],
        *[F.col(col_name).cast("double").alias(col_name) for col_name in DOUBLE_COLS],
        F.trim(F.col("quality_flag")).alias("quality_flag"),
        F.col("source_file_path"),
        F.array_compact(F.array(*[F.when(cast_err(col_name), F.lit(col_name)) for col_name in NUMERIC_COLS]))
            .alias("cast_errors"),
        F.to_json(F.struct(*[F.col(col_name) for col_name in bronze.columns])).alias("raw_payload"),
    )

    geo = spark.table(DIM_GEO).select("geo_key", "cbsa_code")
    staged = typed.join(geo, "cbsa_code", "left")

    step.rows_read = rows_read
    print(f"load_silver_realtor: read {rows_read:,} bronze rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Split good vs quarantined, write quarantine (idempotent: clear this source first), MERGE
# the good rows into the fact, and log the transform. Vars used by both paths are declared
# above the try (CLAUDE.md §11.4).
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = staged.where((F.col("geo_key").isNotNull()) & (F.size("cast_errors") == 0))
    bad  = staged.where((F.col("geo_key").isNull()) | (F.size("cast_errors") > 0))
    rows_rejected = bad.count()

    # Quarantine (no silent drops): clear prior rows for this source, then append current bad.
    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        reason = F.when(F.col("geo_key").isNull(), F.lit("unmatched_geography")) \
                  .otherwise(F.concat(F.lit("cast_failed:"), F.concat_ws(",", F.col("cast_errors"))))
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.col("source_file_path"),
            F.concat_ws("|", F.col("cbsa_code"), F.col("month_date_yyyymm")).alias("natural_key"),
            F.col("raw_payload"),
            reason.alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    fact_cols = ["geo_key", "date_key"] + VALUE_COLS
    good.select(
        *[F.col(col_name) for col_name in fact_cols],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    ).createOrReplaceTempView("realtor_fact_staging")

    set_clause = ", ".join(f"t.{col_name}=s.{col_name}" for col_name in VALUE_COLS) + ", t.updated_ts=s.updated_ts"
    cols_csv   = ", ".join(fact_cols + ["inserted_ts", "updated_ts"])
    vals_csv   = ", ".join(f"s.{col_name}" for col_name in fact_cols + ["inserted_ts", "updated_ts"])
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING realtor_fact_staging s
        ON t.geo_key = s.geo_key AND t.date_key = s.date_key
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({cols_csv}) VALUES ({vals_csv})
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started,
        rows_read=step.rows_read, rows_written=rows_inserted, rows_inserted=rows_inserted,
        rows_updated=rows_updated, rows_rejected=rows_rejected,
        ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_realtor: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise